In [3]:
!pip install -q torch torchvision torchaudio transformers accelerate sentence-transformers faiss-cpu tf-keras

In [4]:
import re
import faiss
import pickle
import torch
import numpy as np

from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

In [5]:
# NEW: S3 dataset download
from sagemaker.s3 import S3Downloader

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [6]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

embedder = SentenceTransformer(
    "BAAI/bge-base-en-v1.5",
    device=device
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
def load_docs_bg(base_path):
    docs = []

    for file in Path(base_path).rglob("chapter*.txt"):
        text = file.read_text(encoding="utf-8", errors="ignore").strip()

        parts = str(file).split("\\")

        try:
            scripture = parts[1]
            canto = [p for p in parts if "canto" in p][0]
            chapter = file.stem.replace("chapter", "")
        except:
            scripture = "unknown"
            canto = "unknown"
            chapter = file.stem

        docs.append({
            "text": text,
            "source": scripture,
            "canto": canto,
            "chapter": chapter,
            "path": str(file)
        })

    return docs

In [8]:
from pathlib import Path

def load_docs_sb(base_path):
    docs = []

    for file in Path(base_path).rglob("chapter*.txt"):

        text = file.read_text(
            encoding="utf-8",
            errors="ignore"
        ).strip()

        parts = file.parts

        try:
            scripture = parts[1]   # sb
            canto = parts[2]       # canto1
            chapter = file.stem.replace("chapter", "")

        except Exception:
            scripture = "unknown"
            canto = "unknown"
            chapter = file.stem

        docs.append({
            "text": text,
            "source": scripture,
            "canto": canto,
            "chapter": chapter,
            "path": str(file)
        })

    return docs

In [9]:
from sagemaker.s3 import S3Downloader

S3Downloader.download(
    "s3://sagemaker-us-east-2-404688195445/dataset/",
    "dataset/"
)
bg_docs = load_docs_bg("dataset/bg")
sb_docs = load_docs_sb("dataset/sb/")

print("BG chapters:", len(bg_docs))
print("SB chapters:", len(sb_docs))

BG chapters: 18
SB chapters: 335


In [10]:
def clean_scripture(text):
    text = text.replace("PURPORT", ". ")
    text = text.replace("Synonyms", ". ")
    text = text.replace("TRANSLATION", ". ")
    return text

In [11]:
def chunk_text(text, chunk_size=800, overlap=100):

    sentences = text.split(". ")

    chunks = []
    current = ""

    for s in sentences:
        if len(current) + len(s) < chunk_size:
            current += s + ". "
        else:
            chunks.append(current.strip())
            current = s + ". "

    if current:
        chunks.append(current.strip())

    return chunks

In [12]:
ENTITY_LIST = [
    "Prahlada",
    "Dhruva",
    "Gajendra",
    "Ambarisha",
    "Hiranyakashipu",
    "Narasimha",
    "Narada",
    "Kapila",
    "Bharata"
]

In [13]:
def extract_entities(text):

    found = []

    for e in ENTITY_LIST:
        if e.lower() in text.lower():
            found.append(e)

    return found

In [14]:
def final_clean(text):
    return text.replace("\n", " ").strip()

In [15]:
import re

def expand_verse_ids(verse_id):
    """
    Converts:
    '16' → ['16']
    '16-18' → ['16','17','18']
    """
    if "-" in verse_id:
        start, end = verse_id.split("-")
        return [str(i) for i in range(int(start), int(end) + 1)]
    return [verse_id]

In [16]:
from pathlib import Path
import re

bg_verses = []

files = list(Path("dataset/bg").rglob("chapter*.txt"))
print("BG files found:", len(files))

for file in files:

    text = file.read_text(encoding="utf-8", errors="ignore")
    text = text.replace("\r", "\n")

    chapter_match = re.search(r"chapter(\d+)", str(file).lower())
    chapter = chapter_match.group(1) if chapter_match else "unknown"

    blocks = re.split(r"(TEXTS?\s+\d+(?:-\d+)?)", text, flags=re.IGNORECASE)

    current_id = None
    current_text = []

    for part in blocks:
        part = part.strip()
        if not part:
            continue

        match = re.match(r"TEXTS?\s+(\d+(?:-\d+)?)", part, re.IGNORECASE)

        if match:
            # flush previous
            if current_id and current_text:
                text_joined = " ".join(current_text).strip()

                # skip tiny garbage chunks
                if len(text_joined) > 30:
                    for vid in expand_verse_ids(current_id):
                        bg_verses.append({
                            "chapter": chapter,
                            "verse_id": vid,
                            "text": text_joined,
                            "entities": extract_entities(text_joined)
                        })

            current_id = match.group(1)
            current_text = []

        else:
            current_text.append(part)

    # flush last block
    if current_id and current_text:
        text_joined = " ".join(current_text).strip()

        if len(text_joined) > 30:
            for vid in expand_verse_ids(current_id):
                bg_verses.append({
                    "chapter": chapter,
                    "verse_id": vid,
                    "text": text_joined,
                    "entities": extract_entities(text_joined)
                })

print("BG verses:", len(bg_verses))

BG files found: 18
BG verses: 700


In [17]:
chunked_sb_docs = []

for doc in sb_docs:

    text = clean_scripture(doc["text"])

    chunks = chunk_text(text)

    for i, chunk in enumerate(chunks):

        chunked_sb_docs.append({
            **doc,
            "chunk_id": i,
            "text": chunk,
            "entities": extract_entities(chunk)
        })

print("SB chunks:", len(chunked_sb_docs))

SB chunks: 24337


In [44]:
import faiss
import pickle

sb_index = faiss.read_index("sb_index.faiss")
bg_index = faiss.read_index("bg_index.faiss")

with open("sb_index_map.pkl", "rb") as f:
    sb_index_map = pickle.load(f)

with open("bg_index_map.pkl", "rb") as f:
    bg_index_map = pickle.load(f)

print("🚀 System ready")

🚀 System ready


In [42]:
#Do no run this step if on the above step you get the result as "System ready"

import numpy as np
import faiss

bg_texts = []
bg_index_map = []

# ---- STEP 1: CLEAN TEXTS ----
for d in bg_verses:
    text = final_clean(d["text"])

    if isinstance(text, str) and len(text.strip()) > 30:
        bg_texts.append(text)
        bg_index_map.append(d)

print("Total BG texts:", len(bg_texts))

# ---- STEP 2: SAFE SETTINGS ----
batch_size = 64   # 🔥 IMPORTANT (was 128 → too large)

dim = None
bg_index = None

# ---- helper ----
def batch_iter(data, size):
    for i in range(0, len(data), size):
        yield data[i:i + size]

# ---- STEP 3: STREAMING EMBEDDING + FAISS ADD ----
for i, batch in enumerate(batch_iter(bg_texts, batch_size)):

    emb = embedder.encode(
        batch,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    # init FAISS once
    if bg_index is None:
        dim = emb.shape[1]
        bg_index = faiss.IndexFlatIP(dim)

    # add directly (NO vstack, NO list storage)
    bg_index.add(np.array(emb).astype("float32"))

    print(f"BG batch {i+1} added | size={len(batch)}")

# ---- FINAL CHECK ----
print("\n✅ BG TEXTS COUNT:", len(bg_texts))
print("✅ FAISS SIZE:", bg_index.ntotal)
print("BG FAISS index built successfully")

Total BG texts: 700
BG batch 1 added | size=64
BG batch 2 added | size=64
BG batch 3 added | size=64
BG batch 4 added | size=64
BG batch 5 added | size=64
BG batch 6 added | size=64
BG batch 7 added | size=64
BG batch 8 added | size=64
BG batch 9 added | size=64
BG batch 10 added | size=64
BG batch 11 added | size=60

✅ BG TEXTS COUNT: 700
✅ FAISS SIZE: 700
BG FAISS index built successfully


In [19]:
#Do no run this step if on the above step you get the result as "System ready"

import numpy as np
import faiss

sb_texts = []
sb_index_map = []

# ---- CLEAN TEXTS ----
for d in chunked_sb_docs:

    text = final_clean(d["text"])

    if isinstance(text, str) and len(text.strip()) > 80:
        sb_texts.append(text)
        sb_index_map.append(d)

print("Total SB texts:", len(sb_texts))

# ---- SAFE SETTINGS ----
batch_size = 64   # 🔥 IMPORTANT (DO NOT INCREASE YET)

def batch_iter(data, size):
    for i in range(0, len(data), size):
        yield data[i:i + size]

sb_index = None
dim = None

# ---- STREAMING EMBEDDING + FAISS ----
for i, batch in enumerate(batch_iter(sb_texts, batch_size)):

    emb = embedder.encode(
        batch,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    emb = np.array(emb).astype("float32")

    if sb_index is None:
        dim = emb.shape[1]
        sb_index = faiss.IndexFlatIP(dim)

    sb_index.add(emb)

    print(f"SB batch {i+1} added | size={len(batch)}")

# ---- FINAL CHECK ----
print("\n✅ SB TEXTS COUNT:", len(sb_texts))
print("✅ FAISS SIZE:", sb_index.ntotal)
print("SB FAISS index built successfully")

Total SB texts: 24329
SB batch 1 added | size=64
SB batch 2 added | size=64
SB batch 3 added | size=64
SB batch 4 added | size=64
SB batch 5 added | size=64
SB batch 6 added | size=64
SB batch 7 added | size=64
SB batch 8 added | size=64
SB batch 9 added | size=64
SB batch 10 added | size=64
SB batch 11 added | size=64
SB batch 12 added | size=64
SB batch 13 added | size=64
SB batch 14 added | size=64
SB batch 15 added | size=64
SB batch 16 added | size=64
SB batch 17 added | size=64
SB batch 18 added | size=64
SB batch 19 added | size=64
SB batch 20 added | size=64
SB batch 21 added | size=64
SB batch 22 added | size=64
SB batch 23 added | size=64
SB batch 24 added | size=64
SB batch 25 added | size=64
SB batch 26 added | size=64
SB batch 27 added | size=64
SB batch 28 added | size=64
SB batch 29 added | size=64
SB batch 30 added | size=64
SB batch 31 added | size=64
SB batch 32 added | size=64
SB batch 33 added | size=64
SB batch 34 added | size=64
SB batch 35 added | size=64
SB batc

In [20]:
faiss.write_index(bg_index, "bg.faiss")
faiss.write_index(sb_index, "sb.faiss")

with open("bg_docs.pkl", "wb") as f:
    pickle.dump(bg_verses, f)

with open("sb_docs.pkl", "wb") as f:
    pickle.dump(chunked_sb_docs, f)

In [22]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32 if device == "cpu" else torch.float16,
    device_map=None
)

llm = llm.to(device)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [23]:
def search_bg(question, k):
    q_emb = embedder.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    # retrieve more candidates for reranking
    scores, ids = bg_index.search(q_emb, k * 10)

    results = []

    #q_lower = question.lower()

    for score, idx in zip(scores[0], ids[0]):

        doc = bg_index_map[idx]

        #text = doc["text"]
        #text_lower = text.lower()

        final_score = float(score)

        # -----------------------------
        # LIGHT DOCTRINE-AWARE BOOST (NO HARD CODING ANSWERS)
        # -----------------------------


        # general Bhagavad Gita instruction boost (Bhagavad Gita conclusion chapters)
        #if doc.get("chapter") in ["18"]:
         #   final_score += 0.3

        results.append({
            "score": final_score,
            "chapter": doc.get("chapter", "unknown"),
            "verse_id": doc.get("verse_id", "unknown"),
            "text": doc["text"]
        })

    # sort by adjusted score
    results.sort(key=lambda x: x["score"], reverse=True)
    
    #print("results:", results)

    #print("Top BG scores:", [round(r["score"], 3) for r in results[:k]])
    #print("Verse:", [r["chapter"] + "." + r["verse_id"] for r in results[:k]])

    return results[:k]

In [24]:
def search_sb(question, k):

    q_emb = embedder.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    scores, ids = sb_index.search(q_emb, k * 3)

    results = []

    #print("\n================ SB RETRIEVAL DEBUG ================\n")

    for rank, (idx, score) in enumerate(zip(ids[0], scores[0])):

        doc = sb_index_map[idx]

        item = {
            "rank": rank + 1,
            "score": float(score),
            "chapter": doc.get("chapter"),
            "canto": doc.get("canto"),
            "text": doc["text"],
            "entities": doc.get("entities", []),
            "path": doc.get("path")
        }

        results.append(item)

        # 🔍 DEBUG VIEW
        #print(f"Rank {rank+1}")
        #print(f"Score: {float(score):.4f}")
        #print(f"Canto: {item['canto']}")
        #print(f"Chapter: {item['chapter']}")
        #print("Text preview:")
        #print(item["text"][:300])
        #print("-" * 60)

    #print("\n================ FINAL RANKING ================\n")

    #for i, r in enumerate(results[:k], 1):
        #print(f"{i}. {r['canto']} | {r['chapter']} | score={r['score']:.4f}")

    return results[:k]

In [25]:
def explain_retrieval(question, results, top_k=3):

    print("========== 📖 SRIMAD BHAGAVATAM RESULTS =============")

    print(f"Query: {question}\n")

    for r in results[:top_k]:

        print(f"Rank {r['rank']}")
        print(f"Similarity Score: {r['score']:.3f}")
        print(f"Source: {r['canto']} | Chapter {r['chapter']}")

        print("Why this was retrieved:")
        
        # simple explanation heuristic
        matched_words = [
            w for w in question.lower().split()
            if w in r["text"].lower()
        ]

        if matched_words:
            print(f"- Keyword overlap: {matched_words}")
        else:
            print("- Semantic similarity (no direct keyword match)")

        #print("\nExcerpt:")
        #print(r["text"][:300])

        print("--------------------------------------------------")

In [26]:
def classify_query(question):
    q = question.lower()

    if any(w in q for w in ["surrender", "take refuge", "submit", "whom shall", "whom should"]):
        return "philosophy"

    if any(w in q for w in ["story", "devotee", "example", "pastime", "incident"]):
        return "narrative"

    return "general"

In [27]:
def clean(text, max_chars=900):
    return text.replace("\n", " ")[:max_chars]

In [28]:
def format_answer(question, bg_results, sb_results):

    bg = bg_results[0]  # best match
    print("All results:", bg_results)

    bg_block = f"""
📜 BHAGAVAD GITA (PRIMARY)

Chapter: {bg.get('chapter', '?')}
Verse: {bg.get('verse_id', '?')}

Text:
{bg['text'][:300]}...

"""

    sb_block = "\n\n".join([
        f"""
📖 SRIMAD BHAGAVATAM

Canto: {s.get('canto', '?')}
Chapter: {s.get('chapter', '?')}

Text:
{s['text'][:250]}...
"""
        for s in sb_results
    ])

    final = f"""
══════════════════════════════════════
QUESTION:
{question}
══════════════════════════════════════

{bg_block}

{sb_block}

══════════════════════════════════════
🎯 FINAL CONCLUSION

Based on Bhagavad Gita and Srimad Bhagavatam:
Lord Krishna is the ultimate shelter (BG 18.66).
══════════════════════════════════════
"""

    return final

In [29]:
def summarize_chunk(text):
    prompt = f"""
Summarize the spiritual teaching of this passage in 1-2 lines.
Focus only on the core instruction or philosophy.

TEXT:
{text}

SUMMARY:
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")

    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=50, do_sample=False)

    return tokenizer.decode(out[0], skip_special_tokens=True)

In [32]:
import torch

def ask(question):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    mode = classify_query(question)

    # -------------------------
    # RETRIEVAL ROUTING
    # -------------------------
    bg_results = search_bg(question, k=2)
    sb_results = search_sb(question, k=2)

    explain_retrieval(question, sb_results)

    bg_context = "\n\n".join(clean(r["text"]) for r in bg_results)
    sb_context = "\n\n".join(clean(r["text"]) for r in sb_results)

    # -------------------------
    # PROMPT
    # -------------------------
    prompt = f"""
You are a spiritual knowledge assistant.

Use ONLY the provided context.

Bhagavad Gita is the primary authority.
Srimad Bhagavatam is supporting explanation.

Do NOT format the answer.
Do NOT add headings or structure.

Just explain the meaning clearly and concisely.

QUESTION:
{question}

BG CONTEXT:
{bg_context}

SB CONTEXT:
{sb_context}

ANSWER:
"""

    # -------------------------
    # TOKENIZE (CPU/GPU SAFE)
    # -------------------------
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    # -------------------------
    # LLM GENERATION
    # -------------------------
    with torch.no_grad():
        out = llm.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False,
            temperature=0.2
        )

    llm_output = tokenizer.decode(out[0], skip_special_tokens=True)

    # -------------------------
    # FORMATTED OUTPUT
    # -------------------------
    bg_block = ""
    if bg_results:
        bg = bg_results[0]
        bg_block = f"""
Chapter: {bg.get('chapter', '?')}
Verse: {bg.get('verse_id', '?')}

Text:
{clean(bg['text'])[:400]}...
"""

    sb_block = "\n\n".join([
        f"""
📖 SRIMAD BHAGAVATAM

Canto: {s.get('canto', '?')}
Chapter: {s.get('chapter', '?')}

Text:
{clean(s['text'])[:250]}...
"""
        for s in sb_results
    ])

    final_output = f"""
========== 📖 BHAGAVADGITA RESULTS =============
{bg_block}

🎯 EXPLANATION (LLM INSIGHT)

{llm_output}

══════════════════════════════════════
"""

    return final_output

In [45]:
import time

#question = "What are the qualities of a devotee?"
question = "What krishna instructs shall we ultimately surrender unto?"
 
start = time.time()

print(ask(question))

print("\nTime taken:", time.time() - start)

========== 📖 SRIMAD BHAGAVATAM RESULTS =============
Query: What krishna instructs shall we ultimately surrender unto?

Rank 1
Similarity Score: 0.751
Source: canto11 | Chapter 11
Why this was retrieved:
- Keyword overlap: ['krishna', 'ultimately', 'surrender']
--------------------------------------------------
Rank 2
Similarity Score: 0.750
Source: canto2 | Chapter 6
Why this was retrieved:
- Keyword overlap: ['krishna', 'surrender']
--------------------------------------------------

========== 📖 BHAGAVADGITA RESULTS =============

Chapter: 18
Verse: 66

Text:
Translation Abandon all varieties of religion and just surrender unto Me. I shall deliver you from all sinful reactions. Do not fear.  Purport The Lord has described various kinds of knowledge and processes of religion – knowledge of the Supreme Brahman, knowledge of the Supersoul, knowledge of the different types of orders and statuses of social life, knowledge of the renounced order of life, kno...


🎯 EXPLANATION (LLM INSIGH

In [34]:
import faiss

faiss.write_index(sb_index, "sb_index.faiss")
print("✅ SB index saved")

✅ SB index saved


In [36]:
faiss.write_index(bg_index, "bg_index.faiss")
print("✅ BG index saved")

✅ BG index saved


In [37]:
import pickle

with open("sb_index_map.pkl", "wb") as f:
    pickle.dump(sb_index_map, f)

print("✅ SB map saved")

✅ SB map saved


In [38]:
with open("bg_index_map.pkl", "wb") as f:
    pickle.dump(bg_index_map, f)

print("✅ BG map saved")

✅ BG map saved


In [40]:
faiss.write_index(sb_index, "sb_index.faiss")
faiss.write_index(bg_index, "bg_index.faiss")

pickle.dump(sb_index_map, open("sb_index_map.pkl", "wb"))
pickle.dump(bg_index_map, open("bg_index_map.pkl", "wb"))